# Milestone 2 parser on Colab (CPU)

Parse the crawled corpus into sharded Parquet tables (`elements`, `hierarchy`, `bibtex`) and generate M2 statistics + performance benchmarks.

1. **Setup** — clone/pull repo, mount Drive safely.
2. **Restore** — unpack `MyDrive/arxiv-data.tar.gz` to Colab **local disk** (`/content/data`) via `restore_data_dir` (never parse on Drive FUSE).
3. **Pilot** — parse 100 papers on 1 worker to check integration and single-core profile.
4. **Full Run** — parse all papers across all vCPUs (`ProcessPoolExecutor`), flushing every 500 papers. Re-running resumes automatically.
5. **Validate & Backup** — run `validate_m2.py` and atomically archive `parsed/` + `stats/` to `MyDrive/arxiv-parsed.tar.gz`.

In [ ]:
import os

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
else:
    !git -C /content/arxiv-pipeline pull --ff-only
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import os
import shutil
import sys
import tarfile
from pathlib import Path

from google.colab import drive

if "/content/arxiv-pipeline/src" not in sys.path:
    sys.path.insert(0, "/content/arxiv-pipeline/src")

from crawler.checkpoint import is_workspace_complete, restore_data_dir

_DRIVE_MOUNT = "/content/drive"
_DATA_DIR = Path("/content/data")
_CRAWL_ARCHIVE = Path("/content/drive/MyDrive/arxiv-data.tar.gz")
_PARSED_ARCHIVE = Path("/content/drive/MyDrive/arxiv-parsed.tar.gz")


def _drive_usable() -> bool:
    my_drive = os.path.join(_DRIVE_MOUNT, "MyDrive")
    if not os.path.isdir(my_drive):
        return False
    try:
        os.listdir(my_drive)
        return True
    except OSError:
        return False


if not _drive_usable():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(_DRIVE_MOUNT) and os.listdir(_DRIVE_MOUNT):
        raise RuntimeError(
            "Cannot mount Drive: /content/drive has stale files. "
            "Use Runtime -> Restart session, then run from the top."
        )
    drive.mount(_DRIVE_MOUNT)

if is_workspace_complete(_DATA_DIR):
    print(f"Local corpus already ready at {_DATA_DIR}.")
else:
    info = restore_data_dir(_CRAWL_ARCHIVE, _DATA_DIR)
    print(f"Restored {_CRAWL_ARCHIVE.name} -> {_DATA_DIR} ({info['final_pairs']} pairs).")


def backup_parsed(*, unmount: bool = False) -> None:
    """Atomically pack parsed/ and stats/ to Google Drive."""
    part = _PARSED_ARCHIVE.with_name(f"{_PARSED_ARCHIVE.name}.part")
    local_tar = Path("/tmp/arxiv-parsed.tar.gz")
    with tarfile.open(local_tar, "w:gz", compresslevel=1) as tf:
        for sub in ("parsed", "stats", "logs"):
            target = _DATA_DIR / sub
            if target.is_dir():
                tf.add(target, arcname=sub)
    with local_tar.open("rb") as fsrc, part.open("wb") as fdst:
        shutil.copyfileobj(fsrc, fdst, length=4 * 1024 * 1024)
        fdst.flush()
        os.fsync(fdst.fileno())
    os.replace(part, _PARSED_ARCHIVE)
    local_tar.unlink(missing_ok=True)
    mib = _PARSED_ARCHIVE.stat().st_size / (1024 * 1024)
    print(f"Saved {_PARSED_ARCHIVE} ({mib:.1f} MiB)")
    if unmount:
        drive.flush_and_unmount()

In [ ]:
# Plan check (no parser modules imported yet)
!python run_parse.py --data-dir /content/data --dry-run

In [ ]:
# Pilot: 100 papers, 1 worker (clean profile of one core)
!python run_parse.py --data-dir /content/data --workers 1 --limit 100

In [ ]:
# Full run: all vCPUs (resumes from finished shards)
import os

!python run_parse.py --data-dir /content/data --workers {os.cpu_count() or 2}

In [ ]:
# Inspect Colab CPU performance metrics (feeds arxiv-book m2/04_parser_performance) & validate
import json
from pathlib import Path

perf_file = Path("/content/data/stats/m2_performance.json")
if perf_file.is_file():
    print(json.dumps(json.loads(perf_file.read_text()), indent=2))
!python validate_m2.py --data-dir /content/data

In [ ]:
# Back up parsed shards + stats to Drive (pass unmount=True before closing tab)
backup_parsed()
# backup_parsed(unmount=True)